# 4. Reporting Visuals

Create export-ready charts that summarize sales, category profitability, and regional performance.

## Environment Setup

Load plotting libraries, project paths, and chart styling for consistent exported visuals.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ASSETS_DIR = PROJECT_ROOT / "assets" / "screenshots"
DATA_DIR.mkdir(exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 150

## Reporting Dataset Load

Import the dataset used for visual summaries and portfolio assets.

In [ ]:
visualization_path = DATA_DIR / "cleaned_data_for_EDA_visualization.csv"
df = pd.read_csv(visualization_path, parse_dates=["Order Date", "Ship Date"])
df.head()

## Formatting Helpers

Define reusable currency labels for clean chart annotations.

In [ ]:
def money_ticks(value, _):
    if abs(value) >= 1_000_000:
        return f"${value / 1_000_000:.1f}M"
    return f"${value / 1_000:.0f}K"

def money_label(value):
    sign = "-" if value < 0 else ""
    value = abs(value)
    if value >= 1_000_000:
        return f"{sign}${value / 1_000_000:.2f}M"
    return f"{sign}${value / 1_000:.1f}K"

## Sales Overview

Create a segment-level sales and profit chart with executive KPI callouts.

In [ ]:
segment = df.groupby("Segment", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
segment_long = segment.melt(id_vars="Segment", value_vars=["Sales", "Profit"], var_name="Metric", value_name="Value")

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=segment_long, x="Segment", y="Value", hue="Metric", palette=["#2f6f9f", "#2ca25f"], ax=ax)
ax.set_title("Sales and Profit by Customer Segment", fontsize=20, weight="bold", pad=18)
ax.set_xlabel("")
ax.set_ylabel("Value")
ax.yaxis.set_major_formatter(FuncFormatter(money_ticks))
for container in ax.containers:
    ax.bar_label(container, labels=[money_label(bar.get_height()) for bar in container], fontsize=10, padding=3)
ax.legend(title="")
ax.text(
    0.01, 0.98,
    f"Total Sales: {money_label(df['Sales'].sum())}\nTotal Profit: {money_label(df['Profit'].sum())}\nMargin: {df['Profit'].sum() / df['Sales'].sum():.2%}",
    transform=ax.transAxes, va="top", ha="left", fontsize=12,
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#d0d7de"),
)
fig.tight_layout()
fig.savefig(ASSETS_DIR / "sales_overview.png", bbox_inches="tight")
plt.show()

## Category Profit View

Create a category profitability chart to highlight margin concentration.

In [ ]:
category = df.groupby("Category", as_index=False).agg(Profit=("Profit", "sum")).sort_values("Profit")

fig, ax = plt.subplots(figsize=(12, 7))
colors = ["#d95f02" if value < 30_000 else "#1b9e77" for value in category["Profit"]]
sns.barplot(data=category, y="Category", x="Profit", hue="Category", palette=colors, legend=False, ax=ax)
ax.set_title("Profit by Product Category", fontsize=20, weight="bold", pad=18)
ax.set_xlabel("Profit")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(FuncFormatter(money_ticks))
for y_position, value in enumerate(category["Profit"]):
    ax.text(value + 3000, y_position, money_label(value), va="center", fontsize=12)
fig.tight_layout()
fig.savefig(ASSETS_DIR / "category_profit.png", bbox_inches="tight")
plt.show()

## Regional Performance View

Create a regional chart comparing sales volume and profit contribution.

In [ ]:
region = df.groupby("Region", as_index=False).agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")).sort_values("Sales", ascending=False)

fig, ax1 = plt.subplots(figsize=(12, 7))
sns.barplot(data=region, x="Region", y="Sales", color="#4c78a8", ax=ax1)
ax1.set_title("Regional Sales and Profit Performance", fontsize=20, weight="bold", pad=18)
ax1.set_xlabel("")
ax1.set_ylabel("Sales")
ax1.yaxis.set_major_formatter(FuncFormatter(money_ticks))
for container in ax1.containers:
    ax1.bar_label(container, labels=[money_label(bar.get_height()) for bar in container], fontsize=10, padding=3)

ax2 = ax1.twinx()
sns.lineplot(data=region, x="Region", y="Profit", marker="o", markersize=10, linewidth=3, color="#e45756", ax=ax2)
ax2.set_ylabel("Profit")
ax2.yaxis.set_major_formatter(FuncFormatter(money_ticks))
for index, row in region.reset_index(drop=True).iterrows():
    ax2.text(index, row["Profit"] + 3500, money_label(row["Profit"]), color="#9d2f2f", ha="center", fontsize=11)
fig.tight_layout()
fig.savefig(ASSETS_DIR / "regional_performance.png", bbox_inches="tight")
plt.show()

## Export Confirmation

List the generated chart assets used in the README.

In [ ]:
print("Exported chart images:")
for chart_path in sorted(ASSETS_DIR.glob("*.png")):
    print(chart_path)